In [0]:
# ========================================
# EXPLORATORY NOTEBOOK
# DATASET: gold_dim_supplier
# ========================================

# ========================================
# 0. CONFIGURATION
# ========================================

CATALOG = "ptfrozenfoods_dev"
SOURCE_SCHEMA = "silver"
TARGET_SCHEMA = "gold"

DOMAIN = "dimensions"
DATASET = "gold_dim_supplier"

STORAGE_ACCOUNT = "stptfrozenfoodsdevwe01"
SILVER_CONTAINER = "silver"
GOLD_CONTAINER = "gold"

SUPPLIERS_DATASET = "erp_suppliers"
SUPPLIERS_TABLE = f"{CATALOG}.{SOURCE_SCHEMA}.{SUPPLIERS_DATASET}"

CANDIDATE_TARGET_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.dim_supplier"

SILVER_BASE_PATH = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
GOLD_BASE_PATH = f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

SUPPLIERS_SOURCE_PATH = f"{SILVER_BASE_PATH}erp/{SUPPLIERS_DATASET}/"
CANDIDATE_TARGET_PATH = f"{GOLD_BASE_PATH}dimensions/dim_supplier/"

In [0]:
# ========================================
# 1. CONTEXT SETUP
# ========================================

print("=" * 80)
print("STARTING GOLD EXPLORATORY: gold_dim_supplier")
print("=" * 80)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SOURCE_SCHEMA}")

print("[INFO] Context setup completed successfully.")

STARTING GOLD EXPLORATORY: gold_dim_supplier
[INFO] Context setup completed successfully.


In [0]:
# ========================================
# 2. CONFIGURATION SUMMARY
# ========================================

print("=" * 80)
print("GOLD EXPLORATORY NOTEBOOK CONFIGURATION")
print("=" * 80)
print(f"Catalog:                         {CATALOG}")
print(f"Source schema:                   {SOURCE_SCHEMA}")
print(f"Target schema:                   {TARGET_SCHEMA}")
print(f"Domain:                          {DOMAIN}")
print(f"Dataset:                         {DATASET}")
print(f"Supliers source table:                  {SUPPLIERS_TABLE}")
print(f"Candidate target table:          {CANDIDATE_TARGET_TABLE}")
print(f"Suppliers source path:            {SUPPLIERS_SOURCE_PATH}")
print(f"Candidate target path:           {CANDIDATE_TARGET_PATH}")
print("=" * 80)

GOLD EXPLORATORY NOTEBOOK CONFIGURATION
Catalog:                         ptfrozenfoods_dev
Source schema:                   silver
Target schema:                   gold
Domain:                          dimensions
Dataset:                         gold_dim_supplier
Supliers source table:                  ptfrozenfoods_dev.silver.erp_suppliers
Candidate target table:          ptfrozenfoods_dev.gold.dim_supplier
Suppliers source path:            abfss://silver@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_suppliers/
Candidate target path:           abfss://gold@stptfrozenfoodsdevwe01.dfs.core.windows.net/dimensions/dim_supplier/


In [0]:
# ========================================
# 3. PRE-CHECKS
# ========================================

print("[INFO] Checking source table availability...")
spark.sql(f"DESCRIBE TABLE {SUPPLIERS_TABLE}")

print("[INFO] Checking source path access...")
dbutils.fs.ls(SUPPLIERS_SOURCE_PATH)

print("[INFO] Checking target container access...")
dbutils.fs.ls(f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/")

print("[INFO] Pre-checks completed successfully.")

[INFO] Checking source table availability...
[INFO] Checking source path access...
[INFO] Checking target container access...
[INFO] Pre-checks completed successfully.


In [0]:
# ========================================
# 4. READ SOURCE DATA
# ========================================

print("[INFO] Loading suppliers from the Silver layer...")

df_suppliers = spark.table(SUPPLIERS_TABLE)

print("[INFO] Source tables loaded successfully.")

print("=" * 80)
print("SOURCE DATA ROW COUNTS")
print("=" * 80)
print(f"Suppliers: {df_suppliers.count():,}")
print("=" * 80)

# ========================================
# DATASET PREVIEW — INITIAL EXPLORATION
# ========================================

print("=" * 80)
print("DATASET PREVIEW — INITIAL EXPLORATION")
print("=" * 80)

print("\n[INFO] Preview: PRODUCTS (df_suppliers)")
print("-" * 80)
display(df_suppliers.limit(5))

print("\n" + "=" * 80)
print("[INFO] Dataset preview completed successfully.")
print("=" * 80)

[INFO] Loading suppliers from the Silver layer...
[INFO] Source tables loaded successfully.
SOURCE DATA ROW COUNTS
Suppliers: 6
DATASET PREVIEW — INITIAL EXPLORATION

[INFO] Preview: PRODUCTS (df_suppliers)
--------------------------------------------------------------------------------


fornecedor_id,nome_fornecedor,pais,status_fornecedor,codigo_legacy,ultima_sincronizacao,load_date,ingestion_timestamp,source_file
F004,DouroHortícola,Portugal,Ativo,LEG-F103,2026-03-15T22:16:51Z,2026-03-17,2026-04-03T22:51:50.272Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_suppliers/load_date=2026-03-17/erp_suppliers.csv
F003,PortoMar Congelados,Portugal,Ativo,LEG-F102,2026-03-15T22:16:51Z,2026-03-17,2026-04-03T22:51:50.272Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_suppliers/load_date=2026-03-17/erp_suppliers.csv
F001,PT Frozen Foods,Portugal,Ativo,LEG-F100,2026-03-15T22:16:51Z,2026-03-17,2026-04-03T22:51:50.272Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_suppliers/load_date=2026-03-17/erp_suppliers.csv
F006,Mar & Serra Foods,Portugal,Ativo,LEG-F105,2026-03-15T22:16:51Z,2026-03-17,2026-04-03T22:51:50.272Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_suppliers/load_date=2026-03-17/erp_suppliers.csv
F005,Doces do Norte,Portugal,Ativo,LEG-F104,2026-03-15T22:16:51Z,2026-03-17,2026-04-03T22:51:50.272Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_suppliers/load_date=2026-03-17/erp_suppliers.csv



[INFO] Dataset preview completed successfully.


In [0]:
# ========================================
# 5. SOURCE VALIDATION
# ========================================

from pyspark.sql import functions as F

print("[INFO] Printing source schema...")
df_suppliers.printSchema()

required_columns = [
    "fornecedor_id"
]

missing_columns = [c for c in required_columns if c not in df_suppliers.columns]

print(f"[INFO] Missing required columns: {missing_columns}")

if missing_columns:
    raise Exception(f"Missing required columns: {missing_columns}")

print("[INFO] Source validation completed successfully.")

[INFO] Printing source schema...
root
 |-- fornecedor_id: string (nullable = true)
 |-- nome_fornecedor: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- status_fornecedor: string (nullable = true)
 |-- codigo_legacy: string (nullable = true)
 |-- ultima_sincronizacao: timestamp (nullable = true)
 |-- load_date: date (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

[INFO] Missing required columns: []
[INFO] Source validation completed successfully.


In [0]:
# ========================================
# 6. INITIAL DATA QUALITY CHECKS
# ========================================

display(
    df_suppliers.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("fornecedor_id").alias("distinct_fornecedor_id"),
        F.sum(F.when(F.col("fornecedor_id").isNull(), 1).otherwise(0)).alias("null_fornecedor_id")
    )
)

total_rows,distinct_fornecedor_id,null_fornecedor_id
6,6,0


In [0]:
# ========================================
# 7. DUPLICATE CHECK — SUPPLIERS
# ========================================

df_duplicates = (
    df_suppliers
    .groupBy("fornecedor_id")
    .count()
    .filter(F.col("count") > 1)
)

display(df_duplicates)

fornecedor_id,count


In [0]:
# ========================================
# 8. BUSINESS PROFILE CHECKS
# ========================================

print("[INFO] Supplier distribution by product count")
display(
    df_suppliers.groupBy("fornecedor_id")
    .count()
    .orderBy(F.col("count").desc())
)

[INFO] Supplier distribution by product count


fornecedor_id,count
F006,1
F005,1
F004,1
F003,1
F002,1
F001,1


In [0]:
# ========================================
# 9. BUILD CANDIDATE DIMENSION
# ========================================

from pyspark.sql import functions as F

df_dim_supplier_candidate = (
    df_suppliers
    .select(
        F.col("fornecedor_id"),
        F.col("nome_fornecedor"),
        F.col("pais"),
        F.col("status_fornecedor"),
        F.col("codigo_legacy"),
        F.col("ultima_sincronizacao"),
        F.col("load_date"),
        F.col("ingestion_timestamp"),
        F.col("source_file")
    )
    .filter(F.col("fornecedor_id").isNotNull())
    .dropDuplicates(["fornecedor_id"])
)

print(f"[INFO] Candidate dimension row count: {df_dim_supplier_candidate.count():,}")
display(df_dim_supplier_candidate)

[INFO] Candidate dimension row count: 6


fornecedor_id,nome_fornecedor,pais,status_fornecedor,codigo_legacy,ultima_sincronizacao,load_date,ingestion_timestamp,source_file
F001,PT Frozen Foods,Portugal,Ativo,LEG-F100,2026-03-15T22:16:51Z,2026-03-17,2026-04-03T22:51:50.272Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_suppliers/load_date=2026-03-17/erp_suppliers.csv
F002,Atlântico Frio Distribuição,Portugal,Ativo,LEG-F101,2026-03-15T22:16:51Z,2026-03-17,2026-04-03T22:51:50.272Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_suppliers/load_date=2026-03-17/erp_suppliers.csv
F003,PortoMar Congelados,Portugal,Ativo,LEG-F102,2026-03-15T22:16:51Z,2026-03-17,2026-04-03T22:51:50.272Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_suppliers/load_date=2026-03-17/erp_suppliers.csv
F004,DouroHortícola,Portugal,Ativo,LEG-F103,2026-03-15T22:16:51Z,2026-03-17,2026-04-03T22:51:50.272Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_suppliers/load_date=2026-03-17/erp_suppliers.csv
F005,Doces do Norte,Portugal,Ativo,LEG-F104,2026-03-15T22:16:51Z,2026-03-17,2026-04-03T22:51:50.272Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_suppliers/load_date=2026-03-17/erp_suppliers.csv
F006,Mar & Serra Foods,Portugal,Ativo,LEG-F105,2026-03-15T22:16:51Z,2026-03-17,2026-04-03T22:51:50.272Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_suppliers/load_date=2026-03-17/erp_suppliers.csv


In [0]:
# ========================================
# 10. CANDIDATE DIMENSION VALIDATION
# ========================================

display(
    df_dim_supplier_candidate.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("fornecedor_id").alias("distinct_fornecedor_id"),
        F.sum(F.when(F.col("fornecedor_id").isNull(), 1).otherwise(0)).alias("null_fornecedor_id")
    )
)

print("[INFO] Candidate dimension validation completed successfully.")

total_rows,distinct_fornecedor_id,null_fornecedor_id
6,6,0


[INFO] Candidate dimension validation completed successfully.


In [0]:
    # ========================================
    # 11. MISSING VALUES ANALYSIS — GENERIC
    # ========================================

    from pyspark.sql import functions as F

    def analyze_missing_values(df, dataset_name):
        print("=" * 80)
        print(f"MISSING VALUES ANALYSIS — {dataset_name.upper()}")
        print("=" * 80)

        total_rows = df.count()

        missing_values_df = df.select([
            F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
            for c in df.columns
        ])

        missing_values_transposed = (
            missing_values_df
            .select(
                F.array([
                    F.struct(
                        F.lit(col_name).alias("column_name"),
                        F.col(col_name).alias("null_count")
                    ) for col_name in missing_values_df.columns
                ]).alias("missing_data")
            )
            .select(F.explode("missing_data").alias("data"))
            .select(
                F.col("data.column_name"),
                F.col("data.null_count")
            )
            .withColumn(
                "null_percentage",
                F.round((F.col("null_count") / F.lit(total_rows)) * 100, 4)
            )
            .orderBy(F.col("null_percentage").desc())
        )

        display(missing_values_transposed)

        print(f"[INFO] Total rows analyzed: {total_rows:,}")
        print("[INFO] Missing values analysis completed successfully.")
        print("=" * 80)

    analyze_missing_values(df_dim_supplier_candidate, "dim_supplier")

MISSING VALUES ANALYSIS — DIM_SUPPLIER


column_name,null_count,null_percentage
status_fornecedor,0,0.0
codigo_legacy,0,0.0
fornecedor_id,0,0.0
load_date,0,0.0
ultima_sincronizacao,0,0.0
nome_fornecedor,0,0.0
ingestion_timestamp,0,0.0
pais,0,0.0
source_file,0,0.0


[INFO] Total rows analyzed: 6
[INFO] Missing values analysis completed successfully.
